# Estudo de técnicas para avaliação de frases
### gramática · estrutura · semântica · coerência

Este notebook é um **estudo comparativo**, não um avaliador. A diferença importa: um avaliador
escolhe uma técnica por dimensão e as integra; um estudo coloca as técnicas **concorrentes** lado
a lado, no mesmo corpus, com o mesmo protocolo, e pergunta qual ganha de qual, em quê, e a que
custo.

Para cada uma das quatro dimensões há:

1. **O mapa** — as técnicas que existem, com fundamentação, o que cada uma mede, quando usar,
   limitações e referência.
2. **A implementação** — 3 a 4 técnicas concorrentes, escritas de forma comparável.
3. **A tabela** — todas rodando no mesmo corpus, com acurácia por tipo de defeito e **custo em
   milissegundos**.
4. **A leitura** — o que a tabela mostra, incluindo onde a técnica mais cara perde para a mais
   barata.

## O protocolo de comparação

Comparar avaliadores exige um corpus com **gabarito**, e a construção certa é o **par mínimo**:
uma frase-base correta e versões dela que mudam **uma coisa só**. Se duas frases diferem apenas na
concordância e a técnica as ordena corretamente, ela está medindo concordância — não comprimento,
vocabulário ou assunto. É a metodologia do CoLA e do BLiMP.

O corpus tem **6 bases × 6 variantes = 36 frases**, mais sondas que não são pares mínimos
(desordem global, frase gramatical sem sentido, frase correta com vocabulário raro) para expor
falsos positivos.

Duas famílias de técnica exigem métricas diferentes, e misturá-las é um erro comum:

| tipo | saída | como se mede | pergunta |
|---|---|---|---|
| **detector binário** | dispara / não dispara | taxa de detecção + falsos positivos | "há erro aqui?" |
| **pontuador contínuo** | um número | acurácia em pares mínimos | "esta frase é melhor que aquela?" |

Um pontuador com "6/6" significa *ordenou o par corretamente*, não *encontrou o erro*. Ele não
localiza nada. As tabelas mantêm as duas famílias separadas por isso.

> **Custo de execução:** o notebook usa spaCy, LanguageTool (Java), Qwen2.5-1.5B e BERTimbau na
> CPU. A carga leva alguns minutos; os testes de embaralhamento (24 permutações × 4 técnicas × 2
> parágrafos) são a parte mais lenta.

In [ ]:
# Dependências (descomente na primeira execução):
# !pip install spacy pyspellchecker language_tool_python transformers torch
# !python -m spacy download pt_core_news_sm

import re, math, time, itertools
from collections import defaultdict, Counter

import spacy, torch 
import torch.nn.functional as F
from spellchecker import SpellChecker
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForMaskedLM

nlp = spacy.load("pt_core_news_sm")
lexico = SpellChecker(language="pt")
TOTAL_TOKENS = lexico.word_frequency.total_words
tokenizar = lambda t: re.findall(r"[a-zà-ÿ]+", t.lower())
print("spaCy + léxico prontos")

spaCy + léxico prontos


In [2]:
# --- CORPUS: 6 bases x 6 variantes, cada variante isolando UM defeito -------------
DEFEITOS = ["correta", "ortografia", "conc_nominal", "conc_verbal", "fragmento", "anomalia"]

PARES = {
 "B1": {
  "correta":      "Os pesquisadores analisaram os dados coletados durante o experimento.",
  "ortografia":   "Os pesquisadores analizaram os dados coletados durante o experimento.",
  "conc_nominal": "Os pesquisadores analisaram os dados coletado durante o experimento.",
  "conc_verbal":  "Os pesquisadores analisou os dados coletados durante o experimento.",
  "fragmento":    "Os pesquisadores que analisaram os dados coletados durante o experimento.",
  "anomalia":     "Os pesquisadores beberam os dados coletados durante o experimento.",
 },
 "B2": {
  "correta":      "A prefeitura instalou novas câmeras nas praças do centro.",
  "ortografia":   "A prefeitura instalou novas câmeras nas praças do sentro.",
  "conc_nominal": "A prefeitura instalou nova câmeras nas praças do centro.",
  "conc_verbal":  "A prefeitura instalaram novas câmeras nas praças do centro.",
  "fragmento":    "A prefeitura que instalou novas câmeras nas praças do centro.",
  "anomalia":     "A prefeitura digeriu novas câmeras nas praças do centro.",
 },
 "B3": {
  "correta":      "Os alunos entregaram os trabalhos antes do prazo final.",
  "ortografia":   "Os alunos entregaram os trabalhos antes do praso final.",
  "conc_nominal": "Os alunos entregaram os trabalho antes do prazo final.",
  "conc_verbal":  "Os alunos entregou os trabalhos antes do prazo final.",
  "fragmento":    "Os alunos que entregaram os trabalhos antes do prazo final.",
  "anomalia":     "Os alunos evaporaram os trabalhos antes do prazo final.",
 },
 "B4": {
  "correta":      "O médico receitou os remédios indicados para a paciente.",
  "ortografia":   "O médico reseitou os remédios indicados para a paciente.",
  "conc_nominal": "O médico receitou os remédios indicado para a paciente.",
  "conc_verbal":  "Os médicos receitou os remédios indicados para a paciente.",
  "fragmento":    "O médico que receitou os remédios indicados para a paciente.",
  "anomalia":     "O médico cantou os remédios indicados para a paciente.",
 },
 "B5": {
  "correta":      "As empresas contrataram profissionais qualificados para o projeto.",
  "ortografia":   "As empresas contrataram proficionais qualificados para o projeto.",
  "conc_nominal": "As empresas contrataram profissionais qualificado para o projeto.",
  "conc_verbal":  "As empresas contratou profissionais qualificados para o projeto.",
  "fragmento":    "As empresas que contrataram profissionais qualificados para o projeto.",
  "anomalia":     "As empresas ferveram profissionais qualificados para o projeto.",
 },
 "B6": {
  "correta":      "Os moradores denunciaram os problemas encontrados no edifício.",
  "ortografia":   "Os moradores denuciaram os problemas encontrados no edifício.",
  "conc_nominal": "Os moradores denunciaram os problema encontrados no edifício.",
  "conc_verbal":  "Os moradores denunciou os problemas encontrados no edifício.",
  "fragmento":    "Os moradores que denunciaram os problemas encontrados no edifício.",
  "anomalia":     "Os moradores assaram os problemas encontrados no edifício.",
 },
}

# Sondas: NÃO são pares mínimos. Expõem desordem global e falso positivo.
SONDAS = {
 "embaralhada_1": "Dados os analisaram pesquisadores os durante coletados experimento o.",
 "embaralhada_2": "Praças instalou câmeras a novas nas prefeitura do centro.",
 "vazia":         "Ideias verdes incolores dormem furiosamente.",
 "rara_correta":  "O sismólogo catalogou anomalias geomagnéticas ininterruptamente.",
}

todas = lambda: [(f"{b}/{t}", b, t, d[t]) for b, d in PARES.items() for t in DEFEITOS]

print(f"{len(todas())} frases em pares mínimos ({len(PARES)} bases x {len(DEFEITOS)} variantes)")
print(f"{len(SONDAS)} sondas\n")
for t in DEFEITOS:
    print(f"  {t:13} {PARES['B1'][t]}")
print("\nsondas:")
for n, f in SONDAS.items():
    print(f"  {n:14} {f}")

36 frases em pares mínimos (6 bases x 6 variantes)
4 sondas

  correta       Os pesquisadores analisaram os dados coletados durante o experimento.
  ortografia    Os pesquisadores analizaram os dados coletados durante o experimento.
  conc_nominal  Os pesquisadores analisaram os dados coletado durante o experimento.
  conc_verbal   Os pesquisadores analisou os dados coletados durante o experimento.
  fragmento     Os pesquisadores que analisaram os dados coletados durante o experimento.
  anomalia      Os pesquisadores beberam os dados coletados durante o experimento.

sondas:
  embaralhada_1  Dados os analisaram pesquisadores os durante coletados experimento o.
  embaralhada_2  Praças instalou câmeras a novas nas prefeitura do centro.
  vazia          Ideias verdes incolores dormem furiosamente.
  rara_correta   O sismólogo catalogou anomalias geomagnéticas ininterruptamente.


In [3]:
# --- PROTOCOLO: as três funções que todas as dimensões vão reusar -----------------

def custo_ms(fn, texto, repeticoes=5):
    # tempo médio por chamada, descartando a primeira (aquecimento)
    fn(texto)
    t0 = time.perf_counter()
    for _ in range(repeticoes):
        fn(texto)
    return (time.perf_counter() - t0) / repeticoes * 1000


def taxa_deteccao(detector):
    # DETECTOR BINÁRIO: quantas vezes dispara, por tipo de defeito.
    # Na linha 'correta', disparar é FALSO POSITIVO.
    c = defaultdict(int)
    for _id, _b, tipo, frase in todas():
        c[tipo] += bool(detector(frase))
    return c


def acuracia_pares(pontuador, maior_melhor=True):
    # PONTUADOR CONTÍNUO: em cada par mínimo, a frase correta vence a corrompida?
    c = defaultdict(int)
    for _b, d in PARES.items():
        base = pontuador(d["correta"])
        for tipo in DEFEITOS[1:]:
            v = pontuador(d[tipo])
            c[tipo] += (base > v) if maior_melhor else (base < v)
    return c


def tabela(titulo, colunas, linhas_por_coluna, total=None):
    print(titulo)
    print(f"{'defeito':14} " + " ".join(f"{c:>13}" for c in colunas))
    for tipo in DEFEITOS if total is None else DEFEITOS[1:]:
        vals = " ".join(f"{linhas_por_coluna[c][tipo]:>10}/6 " for c in colunas)
        marca = ""
        if tipo == "correta" and any(linhas_por_coluna[c][tipo] for c in colunas):
            marca = "   <- FALSO POSITIVO"
        print(f"{tipo:14} {vals}{marca}")
    if total:
        soma = {c: sum(linhas_por_coluna[c][t] for t in DEFEITOS[1:]) for c in colunas}
        print(f"{'TOTAL':14} " + " ".join(f"{soma[c]:>9}/30 " for c in colunas))

print("protocolo definido")

protocolo definido


---
# Dimensão 1 — Gramática

## O mapa das técnicas

### 1. Regras escritas à mão

**Fundamentação.** A gramática normativa já está codificada em livros; um verificador transcreve
essas regras como padrões sobre tokens, lemas e etiquetas morfológicas. "Artigo plural seguido de
substantivo singular" vira um padrão que dispara.

**Mede** violação de regras que um humano previu. **Quando usar:** erros frequentes e bem
delimitados, e sempre que for preciso *explicar a regra* a quem escreveu — nenhuma outra família
entrega "isto viola concordância nominal" como texto pronto. **Limitações:** a cobertura é
exatamente igual ao trabalho humano investido, e varia brutalmente entre línguas; não generaliza
para construções não previstas; falsos positivos em texto literário ou técnico.

**Representantes:** LanguageTool, CoGrOO (português), os verificadores de Word e LibreOffice.
**Referência:** Naber (2003), *A Rule-Based Style and Grammar Checker*.

### 2. Traços morfossintáticos sobre a árvore de dependências

**Fundamentação.** Concordância é, formalmente, **igualdade de traços entre dois nós ligados na
árvore sintática**. Um parser devolve, para cada token, classe, traços (`Number`, `Gender`,
`Person`), função e *head*. Verificar concordância vira percorrer a árvore comparando traços — não
é heurística, é a definição.

**Mede** concordância nominal e verbal, regência. **Quando usar:** línguas de morfologia rica
(português é), e quando se quer **localização exata** do token culpado. **Limitações:** herda todo
erro do parser; e o morfologizador é um modelo estatístico que *prediz* traços em contexto, então
pode absorver o erro que deveria denunciar.

**Representantes:** spaCy, Stanza, UDPipe. **Referência:** Nivre et al. (2016, 2020),
*Universal Dependencies*.

### 3. Aceitabilidade por modelo de linguagem causal

**Fundamentação.** Uma frase agramatical é improvável sob um modelo de linguagem bem treinado.
Basta ler a probabilidade — sem regras, sem anotação, em qualquer língua que o modelo cubra.

O problema é a normalização, e é aqui que mora a literatura. A **perplexidade crua não serve**:
ela confunde *raro* com *errado*, porque palavra improvável derruba a probabilidade tanto quanto
erro de sintaxe. As correções propostas:

| variante | fórmula | corrige |
|---|---|---|
| mean-LP | $\ln P(s)/|s|$ | comprimento |
| PenLP | $\ln P(s)/((5+|s|)/6)^\alpha$ | comprimento (mais suave) |
| **SLOR** | $(\ln P_{LM}(s) - \ln P_{uni}(s))/|s|$ | comprimento **e frequência** |

O SLOR desconta a probabilidade que a frase já teria pela frequência isolada das palavras: o
vocabulário raro é raro nos dois termos e o efeito se cancela.

**Mede** probabilidade estrutural normalizada. **Quando usar:** comparação entre pares mínimos,
triagem em escala. **Limitações:** não localiza nada; a escala não é absoluta (não existe
"SLOR > 4 = bom"); e depende de uma tabela de frequências — palavra fora do vocabulário desregula
o termo unigrama.

**Referência:** Pauls & Klein (2012); Lau, Clark & Lappin (2017), *Grammaticality, Acceptability,
and Probability*.

### 4. Aceitabilidade por modelo mascarado (pseudo-log-likelihood)

**Fundamentação.** BERT não fornece $P(s)$ — ele é treinado para prever tokens mascarados, não
para modelar a sequência. Mas dá para *construir* um escore: mascare cada token, um de cada vez, e
some o log da probabilidade do token original. É a **pseudo-log-likelihood** (PLL).

A vantagem sobre o LM causal é conceitual: cada token é avaliado com **contexto dos dois lados**.
Em "Os pesquisadores ___ os dados", um modelo causal só viu o sujeito; o mascarado vê também o
objeto. Para concordância — que é uma relação bidirecional — isso importa.

**Mede** plausibilidade de cada token no contexto completo. **Quando usar:** quando se quer o
melhor escore de aceitabilidade disponível sem treinar nada, e o custo é aceitável.
**Limitações:** custa **N passes por frase** (um por token) contra 1 do modelo causal; não é
probabilidade normalizada, é um escore; e penaliza vocabulário raro, porque não tem o desconto de
frequência do SLOR.

**Referência:** Wang & Cho (2019); Salazar et al. (2020), *Masked Language Model Scoring*.

### 5. Classificador supervisionado de aceitabilidade

**Fundamentação.** Em vez de inferir aceitabilidade da probabilidade, treinar diretamente na
tarefa: um codificador (BERT/BERTimbau) com uma cabeça de classificação, ajustado em frases
rotuladas como aceitáveis ou não.

**Mede** exatamente o que o conjunto de treino rotulou. **Quando usar:** quando existem dados
rotulados do domínio — é a família que atinge o melhor desempenho quando há dados.
**Limitações:** precisa de anotação; não transfere bem entre domínios; e a métrica correta é o
**coeficiente de correlação de Matthews (MCC)**, não acurácia, porque as classes são
desbalanceadas.

**Referência:** Warstadt, Singh & Bowman (2019), *CoLA: The Corpus of Linguistic Acceptability*.

---

**As quatro implementadas a seguir** cobrem os quatro mecanismos distintos: regra escrita à mão
(T2), estrutura linguística explícita (T1), probabilidade causal (T3) e probabilidade
bidirecional (T4).

In [4]:
# ---- T1: traços morfossintáticos sobre a árvore de dependências (spaCy) ----
def T1_parser(frase):
    doc, erros = nlp(frase), []
    for t in doc:
        n, hn = t.morph.get("Number"), t.head.morph.get("Number")
        if not (n and hn and n != hn):
            continue
        if t.dep_ in ("det", "amod", "acl") and t.head.pos_ in ("NOUN", "PROPN"):
            erros.append(f"nominal: '{t.text}'{n[0]} x '{t.head.text}'{hn[0]}")
        elif t.dep_ in ("nsubj", "nsubj:pass") and t.head.pos_ in ("VERB", "AUX"):
            erros.append(f"verbal: '{t.text}'{n[0]} x '{t.head.text}'{hn[0]}")
    return erros


# ---- T2: regras escritas à mão (LanguageTool pt-BR) ----
import language_tool_python

ferramenta = language_tool_python.LanguageTool("pt-BR")
def T2_languagetool(frase):
    return [m.rule_id for m in ferramenta.check(frase)]


print("T1:", T1_parser(PARES["B1"]["conc_verbal"]))
print("T2:", T2_languagetool(PARES["B1"]["ortografia"]))
print("\nregras pt-BR ativas no LanguageTool:", len(ferramenta._get_languages()) if hasattr(ferramenta, "_get_languages") else "n/d")

T1: ["verbal: 'pesquisadores'Plur x 'analisou'Sing"]
T2: ['MORFOLOGIK_RULE_PT_BR']

regras pt-BR ativas no LanguageTool: 81


In [ ]:
# ---- T3 e T4: aceitabilidade por modelo de linguagem ----
# Dois modelos: um causal (Qwen) e um mascarado (BERTimbau).

QWEN = "Qwen/Qwen2.5-1.5B-Instruct"
tok_q = AutoTokenizer.from_pretrained(QWEN)
mod_q = AutoModelForCausalLM.from_pretrained(QWEN, dtype=torch.float32); mod_q.eval()

BERTIMBAU = "neuralmind/bert-base-portuguese-cased"
tok_b = AutoTokenizer.from_pretrained(BERTIMBAU)
mod_b = AutoModelForMaskedLM.from_pretrained(BERTIMBAU, output_hidden_states=True); mod_b.eval()
print("modelos carregados")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


modelos carregados


In [6]:
def _logprob_causal(frase):
    enc = tok_q(frase, return_tensors="pt"); ids = enc["input_ids"][0]
    with torch.no_grad():
        lp = torch.log_softmax(mod_q(**enc).logits, dim=-1)
    return sum(float(lp[0, i - 1, ids[i]]) for i in range(1, len(ids)))


# ---- T3: SLOR — desconta a frequência isolada das palavras ----
def T3_slor(frase):
    palavras = tokenizar(frase)
    unigrama = sum(math.log((lexico.word_frequency[p] or 1) / TOTAL_TOKENS) for p in palavras)
    return (_logprob_causal(frase) - unigrama) / len(palavras)


# ---- T4: PLL — mascara cada token e soma o log P do original ----
def T4_pll(frase):
    ids = tok_b(frase, return_tensors="pt")["input_ids"][0]
    total = 0.0
    for i in range(1, len(ids) - 1):                  # pula [CLS] e [SEP]
        x = ids.clone(); original = x[i].item(); x[i] = tok_b.mask_token_id
        with torch.no_grad():
            lp = torch.log_softmax(mod_b(x.unsqueeze(0)).logits[0, i], dim=-1)
        total += float(lp[original])
    return total / (len(ids) - 2)


b1 = PARES["B1"]
print(f"{'frase':14} {'SLOR':>8} {'PLL':>8}")
for t in DEFEITOS:
    print(f"{t:14} {T3_slor(b1[t]):8.2f} {T4_pll(b1[t]):8.2f}")

frase              SLOR      PLL
correta            5.82    -0.33
ortografia         5.83    -1.13
conc_nominal       4.91    -1.00
conc_verbal        4.95    -1.14
fragmento          4.93    -1.42
anomalia           3.80    -1.25


In [7]:
# ---- COMPARAÇÃO: as quatro técnicas, mesmo corpus, mesmo protocolo ----
amostra = PARES["B1"]["correta"]

print("--- custo por frase ---")
custos = {}
for nome, fn in [("T1 parser", T1_parser), ("T2 LanguageTool", T2_languagetool),
                 ("T3 SLOR", T3_slor), ("T4 PLL", T4_pll)]:
    custos[nome] = custo_ms(fn, amostra)
    print(f"  {nome:18} {custos[nome]:8.1f} ms")

print("\n--- DETECTORES BINÁRIOS: taxa de disparo por tipo ---")
det = {"T1 parser": taxa_deteccao(T1_parser), "T2 LanguageTool": taxa_deteccao(T2_languagetool)}
tabela("", ["T1 parser", "T2 LanguageTool"], det)

print("\n--- PONTUADORES CONTÍNUOS: acurácia em pares mínimos ---")
pon = {"T3 SLOR": acuracia_pares(T3_slor), "T4 PLL": acuracia_pares(T4_pll)}
tabela("", ["T3 SLOR", "T4 PLL"], pon, total=True)

print("\n--- sondas (não são pares mínimos) ---")
print(f"{'sonda':16} {'T1':>4} {'T2':>4} {'T3 SLOR':>9} {'T4 PLL':>9}")
for n, f in SONDAS.items():
    print(f"{n:16} {len(T1_parser(f)):4} {len(T2_languagetool(f)):4} {T3_slor(f):9.2f} {T4_pll(f):9.2f}")

--- custo por frase ---
  T1 parser               5.7 ms
  T2 LanguageTool       745.0 ms
  T3 SLOR              1023.1 ms
  T4 PLL                569.6 ms

--- DETECTORES BINÁRIOS: taxa de disparo por tipo ---

defeito            T1 parser T2 LanguageTool
correta                 0/6           0/6 
ortografia              0/6           6/6 
conc_nominal            5/6           4/6 
conc_verbal             6/6           1/6 
fragmento               0/6           0/6 
anomalia                0/6           0/6 

--- PONTUADORES CONTÍNUOS: acurácia em pares mínimos ---

defeito              T3 SLOR        T4 PLL
ortografia              5/6           6/6 
conc_nominal            6/6           6/6 
conc_verbal             6/6           6/6 
fragmento               6/6           6/6 
anomalia                6/6           6/6 
TOTAL                 29/30         30/30 

--- sondas (não são pares mínimos) ---
sonda              T1   T2   T3 SLOR    T4 PLL
embaralhada_1       0    0     -0.54  

### Lendo a comparação — Gramática

**Custo.** O parser custa **~4 ms**; as outras três ficam todas na casa das **centenas de
milissegundos** — duas ordens de grandeza acima. A surpresa é que o **LanguageTool custa o mesmo
que os modelos neurais**, apesar de ser baseado em regras. O motivo não é computação, é
arquitetura: `language_tool_python` sobe um servidor Java e faz uma chamada HTTP por frase.
Embutido no processo, seria muito mais rápido.

Duas advertências sobre estes números. Eles **variam entre execuções** — numa medição anterior o
LanguageTool ficou em 690 ms e o SLOR em 668 ms, invertendo a ordem entre os dois. E o que se mede
é *o custo da integração, não o do algoritmo*. Só a diferença de **ordem de grandeza** (4 ms contra
centenas) é conclusão segura; a ordem relativa entre T2, T3 e T4 não é.

**O resultado que inverte a expectativa.** O LanguageTool é a ferramenta madura: 259 MB, décadas
de regras, usada em produção no mundo inteiro. Ele detecta concordância verbal em português
**1 vez em 6**. As ~20 linhas de T1, percorrendo a árvore do spaCy, detectam **6 em 6**.

O inverso vale na ortografia: T2 pega **6/6** (tem dicionário embutido) e T1 pega **0/6** — o
parser não sabe o que é uma palavra inexistente, ele apenas etiqueta o que recebe.

Nenhum dos dois domina. São **complementares**, e a razão é estrutural: o conjunto de regras do
LanguageTool foi construído por contribuidores voluntários, língua a língua, e a cobertura de
português está muito atrás da de inglês ou alemão. Uma técnica cuja qualidade depende de quanto
trabalho humano foi investido *naquela língua* é uma aposta diferente de uma técnica que deriva do
formalismo.

**T3 vs T4 — e por que PLL ganha.** O SLOR acerta **29/30** e o PLL, **30/30**. A única falha do
SLOR é justamente em `ortografia` (5/6), e a causa é conhecida: palavra fora do léxico cai na
contagem-piso, o que **infla** o termo unigrama e faz a frase errada pontuar acima da correta. O
PLL não tem esse problema porque não usa tabela de frequências.

Isso reproduz o achado de Salazar et al. (2020): **contexto bidirecional vence contexto causal**
para aceitabilidade. Faz sentido — concordância é uma relação entre dois pontos da frase, e um
modelo causal só enxerga um lado quando decide o outro.

**Mas o PLL tem um ponto cego grave, e as sondas o expõem.** Olhe `rara_correta` — *"O sismólogo
catalogou anomalias geomagnéticas ininterruptamente"*, uma frase **perfeitamente correta**:

| sonda | SLOR | PLL |
|---|---|---|
| `rara_correta` (correta!) | **+2,01** | **−2,45** |
| `vazia` (sem sentido) | −0,90 | −1,92 |
| `embaralhada_1` (lixo) | −0,54 | −6,85 |
| `embaralhada_2` (lixo) | **+1,43** | −6,89 |

O PLL dá à frase correta uma nota **pior** que a da frase sem sentido, porque pune vocabulário
raro — é exatamente a falha que o SLOR foi inventado para corrigir, e o SLOR de fato a corrige
(+2,01, a melhor nota entre as sondas). Já o SLOR falha onde o PLL brilha: dá **+1,43** a
`embaralhada_2`, uma frase de palavras jogadas ao acaso, enquanto o PLL a afunda em −6,89.

**A conclusão honesta:** **PLL vence dentro do par mínimo; SLOR vence entre frases de conteúdos
diferentes.** Não há vencedor absoluto, e a escolha depende de qual comparação você vai fazer —
o que só fica visível porque o corpus tem sondas além dos pares.

**Todos os quatro são cegos a `fragmento` e `anomalia` como detectores.** T1 e T2 dão 0/6 nos dois.
T3 e T4 "acertam" 6/6, mas só no sentido de ordenar o par — não localizam nem nomeiam. Fragmento
é assunto da Dimensão 2; anomalia, da Dimensão 3.

---
# Dimensão 2 — Estrutura

## O mapa das técnicas

### 1. Teste do núcleo predicativo (detecção de fragmento)

**Fundamentação.** Uma sentença exige um predicado. Numa árvore de dependências, isso é
verificável num ponto só: a **raiz** tem de ser um verbo finito (ou um predicativo com cópula). O
teste ingênuo — "existe algum verbo?" — falha, porque uma oração relativa contém verbo sem
constituir sentença.

**Mede** presença de predicado principal. **Quando usar:** sempre; é barato e binário.
**Limitações:** não diz nada sobre qualidade além da completude.

### 2. Índices de complexidade sintática

**Fundamentação.** Estrutura difícil é estrutura com muito encaixamento. Os índices clássicos:
**profundidade da árvore**, **número de orações**, **taxa de subordinação**, e — sobre árvores de
constituintes — **Yngve** (carga de ramificação à esquerda) e **Frazier** (custo de processamento
incremental).

**Mede** carga estrutural. **Quando usar:** análise de legibilidade e desenvolvimento da escrita
(medir progressão de alunos, comparar registros). **Limitações:** Yngve e Frazier exigem
**árvore de constituintes**, que o `pt_core_news_sm` não fornece — para português seria preciso
outro analisador. E complexidade não é erro: um período longo bem construído é legítimo.

**Referência:** Yngve (1960); Frazier (1985); Lu (2010) para índices automatizados.

### 3. Comprimento de dependência e não-projetividade

**Fundamentação.** Línguas tendem a **minimizar a distância entre palavras ligadas** — é um
princípio de eficiência de processamento com forte suporte translinguístico. Arcos longos custam
memória de trabalho. E **arcos que se cruzam** (não-projetividade) marcam construções deslocadas.

**Mede** esforço de processamento estrutural. **Quando usar:** comparação entre registros e
estudos psicolinguísticos. **Limitações:** é uma tendência estatística sobre corpora, não um
critério de correção; frases curtas quase não variam.

**Referência:** Futrell, Mahowald & Gibson (2015), *Large-scale evidence of dependency length
minimization*; Gibson (1998).

### 4. Fórmulas de legibilidade

**Fundamentação.** Regressões ajustadas em meados do século XX para prever a série escolar
necessária para compreender um texto, a partir de duas variáveis de superfície: comprimento da
frase e tamanho das palavras.

Para o português, a adaptação do Flesch (Martins et al., 1996; base das ferramentas do NILC/USP):

$$\text{ILF} = 248{,}835 - 1{,}015 \times \frac{\text{palavras}}{\text{frases}} - 84{,}6 \times \frac{\text{sílabas}}{\text{palavras}}$$

A constante muda em relação ao original (206,835) porque a palavra portuguesa é mais longa. O
**Gunning Fog** usa outra combinação: comprimento da frase e proporção de palavras polissilábicas.

**Mede** densidade lexical e comprimento — **e nada mais**. **Quando usar:** triagem de
adequação a um público, em textos longos. **Limitações:** foram calibradas para **textos**, não
frases; não leem nada; e são trivialmente enganáveis — trocar termos precisos por perífrases
vagas melhora o índice e piora o texto.

**Referência:** Flesch (1948); Martins et al. (1996); Gunning (1952).

### 5. Coh-Metrix e Coh-Metrix-Port

Não é uma técnica, é um **pacote de ~100 índices** cobrindo legibilidade, complexidade sintática,
coesão referencial e diversidade lexical, com adaptação para o português brasileiro feita pelo
NILC/USP. É a referência de partida para quem for construir métricas estruturais em pt.

**Referência:** Graesser et al. (2004); Scarton & Aluísio (2010) para o Coh-Metrix-Port.

In [8]:
# ---- E1: teste do núcleo predicativo ----
def E1_fragmento(frase):
    doc = nlp(frase)
    raiz = [t for t in doc if t.dep_ == "ROOT"][0]
    finito = raiz.pos_ in ("VERB", "AUX") or any(f.dep_ == "cop" for f in raiz.children)
    return [] if finito else [f"fragmento: raiz '{raiz.text}' é {raiz.pos_}, não verbo"]


# ---- E2: índices de complexidade da árvore ----
SUBORD = ("acl", "acl:relcl", "advcl", "ccomp", "xcomp", "csubj")

def E2_complexidade(frase):
    doc = nlp(frase)
    sem_pont = [t for t in doc if t.pos_ != "PUNCT"]
    return {
        "profundidade": max(len(list(t.ancestors)) for t in doc),
        "oracoes":      1 + sum(1 for t in doc if t.dep_ in SUBORD),
        "ramificacao":  sum(len(list(t.children)) for t in sem_pont) / max(1, len(sem_pont)),
    }


# ---- E3: comprimento de dependência e arcos cruzados ----
def E3_dependencia(frase):
    doc = nlp(frase)
    arcos = [(min(t.i, t.head.i), max(t.i, t.head.i))
             for t in doc if t.head != t and t.pos_ != "PUNCT"]
    if not arcos:
        return {"comprimento": 0.0, "cruzamentos": 0}
    cruz = sum(1 for i, (a, b) in enumerate(arcos) for (c, d) in arcos[i + 1:]
               if a < c < b < d or c < a < d < b)
    return {"comprimento": sum(b - a for a, b in arcos) / len(arcos), "cruzamentos": cruz}


# ---- E4: legibilidade ----
VOGAIS = "aeiouáéíóúâêôàãõ"

def silabas(palavra):
    n, anterior = 0, False
    for c in palavra.lower():
        v = c in VOGAIS
        n += v and not anterior
        anterior = v
    return max(1, n)

def E4_legibilidade(texto):
    frases_ = [s for s in re.split(r"[.!?]+", texto) if s.strip()]
    pal = tokenizar(texto)
    ppf = len(pal) / len(frases_)
    spp = sum(silabas(p) for p in pal) / len(pal)
    polis = sum(1 for p in pal if silabas(p) >= 3) / len(pal)
    return {"flesch": 248.835 - 1.015 * ppf - 84.6 * spp, "fog": 0.4 * (ppf + 100 * polis)}


print(f"{'frase':14} {'fragmento?':>11} {'prof':>5} {'orac':>5} {'ramif':>6} {'dep.len':>8} {'cruz':>5} {'Flesch':>8} {'Fog':>6}")
for t in DEFEITOS:
    f = PARES["B1"][t]
    c, d, l = E2_complexidade(f), E3_dependencia(f), E4_legibilidade(f)
    print(f"{t:14} {str(bool(E1_fragmento(f))):>11} {c['profundidade']:5} {c['oracoes']:5} "
          f"{c['ramificacao']:6.2f} {d['comprimento']:8.2f} {d['cruzamentos']:5} {l['flesch']:8.1f} {l['fog']:6.1f}")

frase           fragmento?  prof  orac  ramif  dep.len  cruz   Flesch    Fog
correta              False     4     2   1.00     1.50     0    -14.1   25.8
ortografia           False     4     2   1.00     1.50     0    -14.1   25.8
conc_nominal         False     4     2   1.00     1.50     0    -14.1   25.8
conc_verbal          False     4     2   1.00     1.50     0     -4.7   25.8
fragmento             True     5     3   1.00     1.56     0      1.8   24.0
anomalia             False     4     2   1.00     1.50     0      4.7   25.8


In [9]:
# ---- COMPARAÇÃO: Estrutura ----
amostra = PARES["B1"]["correta"]

print("--- custo por frase ---")
for nome, fn in [("E1 fragmento", E1_fragmento), ("E2 complexidade", E2_complexidade),
                 ("E3 dependência", E3_dependencia), ("E4 legibilidade", E4_legibilidade)]:
    print(f"  {nome:18} {custo_ms(fn, amostra, 20):8.2f} ms")

print("\n--- E1 como DETECTOR BINÁRIO ---")
tabela("", ["E1 fragmento"], {"E1 fragmento": taxa_deteccao(E1_fragmento)})

print("\n--- E2/E3/E4 como PONTUADORES: médias por tipo ---")
ag = defaultdict(lambda: defaultdict(list))
for _i, _b, tipo, f in todas():
    c, d, l = E2_complexidade(f), E3_dependencia(f), E4_legibilidade(f)
    for k, v in [("prof", c["profundidade"]), ("orac", c["oracoes"]), ("ramif", c["ramificacao"]),
                 ("dep", d["comprimento"]), ("cruz", d["cruzamentos"]),
                 ("flesch", l["flesch"]), ("fog", l["fog"])]:
        ag[tipo][k].append(v)
print(f"{'defeito':14} {'prof':>6} {'orações':>8} {'ramif':>7} {'dep.len':>8} {'cruz':>6} {'Flesch':>8} {'Fog':>7}")
for t in DEFEITOS:
    m = {k: sum(v) / len(v) for k, v in ag[t].items()}
    print(f"{t:14} {m['prof']:6.1f} {m['orac']:8.1f} {m['ramif']:7.2f} {m['dep']:8.2f} "
          f"{m['cruz']:6.1f} {m['flesch']:8.1f} {m['fog']:7.1f}")

print("\n--- acurácia em pares mínimos ---")
pon = {"E3 dep.len": acuracia_pares(lambda f: E3_dependencia(f)["comprimento"], maior_melhor=False),
       "E4 Flesch":  acuracia_pares(lambda f: E4_legibilidade(f)["flesch"])}
tabela("", ["E3 dep.len", "E4 Flesch"], pon, total=True)

print("\n--- sondas ---")
print(f"{'sonda':16} {'fragmento?':>11} {'prof':>5} {'dep.len':>8} {'cruz':>5} {'Flesch':>9}")
for n, f in SONDAS.items():
    c, d, l = E2_complexidade(f), E3_dependencia(f), E4_legibilidade(f)
    print(f"{n:16} {str(bool(E1_fragmento(f))):>11} {c['profundidade']:5} "
          f"{d['comprimento']:8.2f} {d['cruzamentos']:5} {l['flesch']:9.1f}")

--- custo por frase ---
  E1 fragmento           6.82 ms
  E2 complexidade        5.32 ms
  E3 dependência         5.80 ms
  E4 legibilidade        0.05 ms

--- E1 como DETECTOR BINÁRIO ---

defeito         E1 fragmento
correta                 0/6 
ortografia              0/6 
conc_nominal            0/6 
conc_verbal             0/6 
fragmento               6/6 
anomalia                0/6 

--- E2/E3/E4 como PONTUADORES: médias por tipo ---
defeito          prof  orações   ramif  dep.len   cruz   Flesch     Fog
correta           3.8      1.7    1.00     1.43    0.0     26.0    23.7
ortografia        3.8      1.7    1.00     1.43    0.0     26.0    23.7
conc_nominal      3.7      1.7    1.00     1.47    0.0     26.0    23.7
conc_verbal       4.0      1.7    1.00     1.39    0.0     31.1    23.7
fragmento         4.8      2.7    1.00     1.50    0.0     38.5    21.9
anomalia          3.8      1.7    1.00     1.43    0.0     32.7    22.9

--- acurácia em pares mínimos ---

defeito       

### Lendo a comparação — Estrutura

**E1 é o melhor detector deste notebook inteiro.** 6/6 no alvo, **0 falsos positivos** em todas as
outras 30 frases, ~4 ms, binário, e com localização (aponta a raiz). Nenhuma outra técnica de
nenhuma dimensão alcança essa combinação. A razão é que ele não é uma heurística: "toda sentença
tem predicado" é uma verdade sintática, e a árvore expõe o predicado num nó só.

Repare que `fragmento` tem **uma oração a mais** (2,7 contra 1,7 em média) e é **mais profundo**
(4,8 contra 3,8): o `que` rebaixou o verbo principal a oração relativa. As duas métricas detectam
o mesmo evento — mas E1 responde sim/não, e as de E2 respondem com um número que precisa de
limiar.

**E2 tem um índice que é pura ilusão: `ramificacao` dá 1,00 em absolutamente tudo.** Não é
coincidência nem falta de sensibilidade — é uma **identidade matemática**. Numa árvore com *n* nós
há *n−1* arcos, e "média de filhos por nó" é (n−1)/n ≈ 1 sempre, para qualquer frase de qualquer
qualidade. Uma métrica que não pode variar não é uma métrica. É o tipo de índice que sobrevive em
relatórios por parecer sofisticado, e o teste que o mata custa uma coluna numa tabela.

**E3 é redundante e o cruzamento é inerte.** O comprimento de dependência acerta 6/6 em
`fragmento` — exatamente onde E1 já acertava, sendo mais caro e sem veredito binário. Nos demais
defeitos, fica em 0–1/6. E **`cruzamentos` dá 0 em todo o corpus, inclusive nas embaralhadas**:
não-projetividade exige construções genuinamente deslocadas, que frases SVO curtas em português
não produzem. Métrica correta, corpus errado — e reportar "0 cruzamentos" como se fosse evidência
de boa estrutura seria um erro de leitura.

**E4 é o pior resultado do estudo: a legibilidade está anticorrelacionada com a correção.**

| frase | Flesch | veredito humano |
|---|---|---|
| `fragmento` (média) | **38,5** | quebrada |
| `correta` (média) | 26,0 | correta |
| `embaralhada_2` | **61,1** | lixo |
| `rara_correta` | **−109,8** | correta |

A frase quebrada é "mais fácil" que a correta; a de palavras embaralhadas tira a melhor nota do
conjunto; e uma frase perfeita com vocabulário técnico tira **−109,8**, muito fora da escala de
0–100. Em pares mínimos, o Flesch fica em **0–1/6** — no nível do acaso ou abaixo.

Nada disso é defeito de implementação. A fórmula só olha *comprimento de frase* e *sílabas por
palavra*; `fragmento` é mais curto que a base, e `rara_correta` é polissilábica. **Ela mede o que
diz medir; o erro é usá-la para outra coisa.** Fórmulas de legibilidade respondem "que escolaridade
este texto exige", não "este texto está certo" — e valem para textos longos, não para frases.

**Custo:** E4 é **centenas de vezes** mais barata que as demais (~0,01 ms contra ~3,5 ms) porque é
aritmética pura, sem parser nenhum. É simultaneamente a técnica mais barata e a menos informativa
do estudo — um lembrete de que custo baixo, sozinho, não é argumento a favor.

---
# Dimensão 3 — Semântica

Aqui a frase já passou por ortografia, concordância e estrutura. *"Os pesquisadores **beberam** os
dados"* tem árvore impecável e concordância perfeita. O que está errado é uma **restrição de
seleção**: *beber* exige paciente líquido.

## O mapa das técnicas

### 1. Surpresa por token

**Fundamentação.** $-\ln P(\text{token} \mid \text{contexto})$ — o modelo já calcula isso. Onde a
frase viola uma expectativa, a surpresa dispara **naquele token**.

**Mede** improbabilidade local. **Quando usar:** quando se quer localização barata a partir de um
modelo já carregado. **Limitações:** o primeiro token é sempre um pico falso (não há contexto);
e surpresa alta confunde **anômalo** com **raro**, **original** e **bem escrito**.

### 2. Preenchimento mascarado / restrições de seleção

**Fundamentação.** Em vez de perguntar "quão provável é esta frase", perguntar **"o que caberia
aqui?"**. Mascare o verbo e leia a distribuição: se o verbo escrito está fora do que o modelo
esperava, é violação de restrição de seleção. É a formalização computacional de *thematic fit*.

**Mede** compatibilidade entre predicado e argumentos. **Quando usar:** quando o defeito esperado
é troca de palavra em posição estrutural conhecida. **Limitações:** exige saber **onde** mascarar
(precisa do parser); não se aplica se a posição não existir (numa frase sem verbo raiz, por
exemplo); e o top-k é uma amostra da distribuição, não um veredito.

**Referência:** Erk (2007); Sasano & Korhonen (2020) sobre *selectional preference* neural.

### 3. Embeddings e similaridade distribucional

**Fundamentação.** Hipótese distribucional: palavras que ocorrem em contextos parecidos têm
significados parecidos. Frases viram vetores; comparações viram cossenos.

**Mede** proximidade temática. **Quando usar:** busca, agrupamento, deduplicação, "esta resposta
fala do assunto certo?". **Limitações:** o ponto cego clássico é a **negação** — "cresceu" e "não
cresceu" ficam quase colados; e a média de camada de um modelo grande comprime tudo numa faixa
estreita, onde só a *ordem* tem alguma informação.

**Referência:** Harris (1954); Reimers & Gurevych (2019) para SBERT.

### 4. Inferência de linguagem natural (NLI)

**Fundamentação.** Treinar um modelo para decidir se uma sentença **implica**, **contradiz** ou é
**neutra** em relação a outra. Com isso dá para testar plausibilidade e contradição interna de
forma explícita, em vez de inferir de probabilidade.

**Mede** relação lógica entre proposições. **Quando usar:** verificação factual contra uma fonte,
detecção de contradição, avaliação de coerência semântica. **Limitações:** exige um modelo NLI
treinado (para português, os recursos vêm do **ASSIN/ASSIN2**); e o desempenho cai fora do domínio
de treino.

**Referência:** Bowman et al. (2015), SNLI; Real et al. (2020), ASSIN 2.

### 5. Rotulagem de papéis semânticos (SRL) e desambiguação de sentido (WSD)

Camadas de análise linguística explícita — *quem fez o quê a quem* — que permitem verificar
restrições de seleção de forma simbólica em vez de estatística. Mais interpretáveis, mais caras de
obter, e com cobertura menor em português.

### 6. LLM como juiz de plausibilidade

Perguntar diretamente. Cobre tudo, explica em linguagem natural, e é o mais caro e o menos
reprodutível. Sensível ao formato do prompt.

---

As quatro implementadas: surpresa (1), verbo mascarado (2), embeddings (3) e juízo por LLM (6).

In [10]:
# ---- S1: pico de surpresa (LM causal), descartando o 1o token real ----
def S1_surpresa(frase):
    enc = tok_q(frase, return_tensors="pt"); ids = enc["input_ids"][0]
    with torch.no_grad():
        lp = torch.log_softmax(mod_q(**enc).logits, dim=-1)
    vals = [(tok_q.decode(ids[i]), -float(lp[0, i - 1, ids[i]])) for i in range(2, len(ids))]
    return max(vals, key=lambda x: x[1])


# ---- S2: plausibilidade do verbo por preenchimento mascarado ----
def S2_verbo_mascarado(frase):
    doc = nlp(frase)
    raiz = [t for t in doc if t.dep_ == "ROOT" and t.pos_ == "VERB"]
    if not raiz:                       # sem verbo na raiz, a técnica NÃO SE APLICA
        return None
    alvo = raiz[0].text
    enc = tok_b(frase.replace(alvo, tok_b.mask_token, 1), return_tensors="pt")
    pos = (enc["input_ids"][0] == tok_b.mask_token_id).nonzero()
    if len(pos) == 0:
        return None
    with torch.no_grad():
        lp = torch.log_softmax(mod_b(**enc).logits[0, int(pos[0])], dim=-1)
    ids_alvo = tok_b(alvo, add_special_tokens=False)["input_ids"]
    return {"verbo": alvo, "logp": float(lp[ids_alvo[0]]),
            "esperados": [tok_b.decode(i) for i in lp.topk(5).indices.tolist()]}


# ---- S3: embedding da frase (BERTimbau, média da última camada) ----
def embedding(frase):
    enc = tok_b(frase, return_tensors="pt")
    with torch.no_grad():
        return mod_b(**enc).hidden_states[-1][0].mean(dim=0)

def S3_cosseno(a, b):
    return float(F.cosine_similarity(embedding(a), embedding(b), dim=0))


# ---- S4: juízo de plausibilidade por LLM ----
def S4_llm(frase):
    p = ("Responda apenas SIM ou NAO.\n"
         "A frase abaixo descreve algo fisicamente possível e faz sentido?\n"
         f"Frase: {frase}\nResposta:")
    enc = tok_q(p, return_tensors="pt")
    with torch.no_grad():
        o = mod_q.generate(**enc, max_new_tokens=4, do_sample=False, pad_token_id=tok_q.eos_token_id)
    saida = tok_q.decode(o[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return saida.split()[0][:6] if saida.split() else "?"


print("--- S2: o que o modelo esperava naquela posição? ---")
for b in ["B1", "B3", "B5"]:
    for t in ["correta", "anomalia"]:
        r = S2_verbo_mascarado(PARES[b][t])
        print(f"  {b}/{t:9} '{r['verbo']:12}' logP = {r['logp']:7.2f}   esperava: {r['esperados']}")

--- S2: o que o modelo esperava naquela posição? ---
  B1/correta   'analisaram  ' logP =   -5.39   esperava: ['usaram', 'consideraram', 'apresentaram', 'usam', 'consideram']
  B1/anomalia  'beberam     ' logP =  -16.29   esperava: ['usaram', 'consideraram', 'apresentaram', 'usam', 'consideram']
  B3/correta   'entregaram  ' logP =   -6.70   esperava: ['apresentaram', 'iniciaram', 'apresentam', 'terminaram', 'começaram']
  B3/anomalia  'evaporaram  ' logP =  -18.69   esperava: ['apresentaram', 'iniciaram', 'apresentam', 'terminaram', 'começaram']
  B5/correta   'contrataram ' logP =   -6.71   esperava: ['buscam', 'procuram', 'possuem', 'exigem', 'oferecem']
  B5/anomalia  'ferveram    ' logP =  -19.55   esperava: ['buscam', 'procuram', 'possuem', 'exigem', 'oferecem']


In [11]:
# ---- COMPARAÇÃO: Semântica ----
amostra = PARES["B1"]["correta"]

print("--- custo por frase ---")
for nome, fn in [("S1 surpresa", S1_surpresa), ("S2 verbo mascarado", S2_verbo_mascarado),
                 ("S3 embedding", embedding), ("S4 LLM", S4_llm)]:
    print(f"  {nome:20} {custo_ms(fn, amostra, 3):8.1f} ms")

print("\n--- S1/S2: médias por tipo de defeito ---")
ag = defaultdict(lambda: {"s1": [], "s2": []})
for _i, _b, tipo, f in todas():
    ag[tipo]["s1"].append(S1_surpresa(f)[1])
    r = S2_verbo_mascarado(f)
    if r: ag[tipo]["s2"].append(r["logp"])
print(f"{'defeito':14} {'S1 pico (nats)':>15} {'S2 logP(verbo)':>16}")
for t in DEFEITOS:
    s2 = ag[t]["s2"]
    v2 = f"{sum(s2)/len(s2):16.2f}" if s2 else f"{'n/a':>16}"
    print(f"{t:14} {sum(ag[t]['s1'])/len(ag[t]['s1']):15.2f} {v2}")
print("  (S2 = n/a em 'fragmento': não há verbo na raiz para mascarar)")

print("\n--- acurácia: a técnica aponta ANOMALIA como a pior das 6 variantes? ---")
a1 = a2 = 0
for _b, d in PARES.items():
    p1 = {t: S1_surpresa(d[t])[1] for t in DEFEITOS}
    p2 = {t: S2_verbo_mascarado(d[t]) for t in DEFEITOS}
    p2 = {t: r["logp"] for t, r in p2.items() if r}
    a1 += max(p1, key=p1.get) == "anomalia"
    a2 += min(p2, key=p2.get) == "anomalia"
print(f"  S1 (maior surpresa = anomalia):   {a1}/6")
print(f"  S2 (menor logP verbo = anomalia): {a2}/6")

print("\n--- S3: cosseno de cada variante contra a base correta ---")
for t in DEFEITOS[1:]:
    v = [S3_cosseno(PARES[b]["correta"], PARES[b][t]) for b in PARES]
    print(f"  {t:14} {sum(v)/len(v):.4f}")
afirm = "A área monitorada cresceu no período."
negac = "A área monitorada não cresceu no período."
print(f"  {'negação':14} {S3_cosseno(afirm, negac):.4f}   <- sentidos OPOSTOS")

print("\n--- S4: juízo do LLM ---")
ok = 0
for b, d in PARES.items():
    rc, ra = S4_llm(d["correta"]), S4_llm(d["anomalia"])
    acerto = rc.upper().startswith("SIM") and not ra.upper().startswith("SIM")
    ok += acerto
    print(f"  {b}: correta='{rc}'  anomalia='{ra}'  {'ok' if acerto else '<- ERROU'}")
print(f"  acertou os dois: {ok}/6")

--- custo por frase ---
  S1 surpresa            1025.0 ms
  S2 verbo mascarado       57.6 ms
  S3 embedding             53.0 ms
  S4 LLM                 2520.4 ms

--- S1/S2: médias por tipo de defeito ---
defeito         S1 pico (nats)   S2 logP(verbo)
correta                   7.72            -5.44
ortografia               10.75            -9.54
conc_nominal              8.60            -4.92
conc_verbal               8.45            -8.69
fragmento                 7.68              n/a
anomalia                 13.20           -18.16
  (S2 = n/a em 'fragmento': não há verbo na raiz para mascarar)

--- acurácia: a técnica aponta ANOMALIA como a pior das 6 variantes? ---
  S1 (maior surpresa = anomalia):   4/6
  S2 (menor logP verbo = anomalia): 5/6

--- S3: cosseno de cada variante contra a base correta ---
  ortografia     0.9434
  conc_nominal   0.9766
  conc_verbal    0.9589
  fragmento      0.9518
  anomalia       0.9282
  negação        0.9660   <- sentidos OPOSTOS

--- S4: juíz

### Lendo a comparação — Semântica

**S2 é a melhor técnica desta dimensão, e a saída explica por quê.** Mascarando o verbo em
*"Os pesquisadores ___ os dados coletados"*, o modelo esperava `usaram`, `consideraram`,
`apresentaram` — e atribui **−5,39** ao `analisaram` que estava escrito contra **−16,29** ao
`beberam`. Onze nats de diferença, no token exato, com a lista do que *deveria* estar ali.

O detalhe metodológico bonito: o top-5 é **idêntico** nas duas frases. Só o token mascarado muda,
o contexto é o mesmo, então a expectativa do modelo é a mesma — o que muda é o quanto a palavra
escrita se encaixa nela. Isso é *thematic fit* medido de forma direta, e a acurácia é **5/6**.

**S1 chega a 4/6 — e é cerca de 14× mais CARA que S2** (~520 ms contra ~38 ms). Não há troca aqui:
S2 é mais exata *e* mais barata. O motivo é que S1 roda o Qwen inteiro sobre a frase para varrer
todos os tokens, enquanto S2 faz **um** passe num modelo bem menor, na única posição que interessa.
E ela sabe qual é essa posição porque **o parser lhe diz** — é a única técnica do estudo em que uma
camada linguística e uma estatística se combinam, e é a que ganha.

**Mas nem S1 nem S2 são específicos de semântica.** Repare na linha `ortografia`: S1 = 10,75, a
segunda maior surpresa do corpus, atrás apenas de `anomalia` (13,20); e S2 = −9,54, também baixo.
Faz sentido — uma palavra inexistente é imprevisível tanto quanto um verbo impossível. As duas
medem **improbabilidade**, e improbabilidade não informa a causa. É por isso que a acurácia aqui é
medida como "a anomalia é a **pior das seis**" e não "a anomalia dispara": disparar, ela dispara
para vários defeitos.

**A limitação de S2 aparece na linha `fragmento`: n/a.** Sem verbo na raiz, não há o que mascarar.
A técnica não erra — ela **não se aplica**, e é importante que a tabela mostre `n/a` em vez de um
zero. Zero é um valor; ausência não é. Na primeira versão deste código a função devolvia 0,0
nesse caso, e a média por tipo ficava contaminada por um número que não significava nada.

**S3 (embeddings) mal discrimina, e a ordem que produz não é confiável.** Todos os cossenos ficam
entre 0,93 e 0,98. `anomalia` é de fato o mais distante (0,9282) — mas:

- **negação** marca **0,9660**, *mais similar* que a anomalia. Uma frase que afirma o **oposto**
  parece mais próxima do original do que uma com verbo estranho.
- **`ortografia` (0,9434) fica mais distante que `conc_nominal` (0,9766)**: um erro de digitação
  desloca mais o vetor que um erro de gramática, porque quebra a subtokenização.

Ou seja: a ordem existe, mas não corresponde à gravidade linguística. Embeddings servem para
**assunto**, não para **correção** — usá-los como detector de erro é usar a ferramenta errada.

**S4 (LLM) acerta 4/6 e erra de forma reveladora.** Os dois erros são `B5` ("As empresas
**ferveram** profissionais") e `B6` ("Os moradores **assaram** os problemas") — justamente os dois
verbos com leitura idiomática possível em português ("ferver de gente", "assar" como gíria). O
modelo não está claramente errado; ele está aceitando uma leitura figurada. Isso expõe o limite
real da dimensão: **anomalia semântica e metáfora são a mesma coisa vista de ângulos diferentes**,
e nenhuma técnica aqui separa as duas sem contexto.

E S4 é de longe a mais cara das quatro (~1 600 ms, mais de 40× o S2), para empatar com S1 em 4/6.

---
# Dimensão 4 — Coesão e coerência

Esta dimensão **não cabe numa frase**. "O carro é azul" não é coerente nem incoerente — coerência
é relação *entre* frases. Duas noções distintas:

- **Coesão**: os elos de superfície — repetição, pronomes, conectivos, cadeias de referência.
- **Coerência**: o texto se sustenta como um todo e as ideias progridem.

Coesão é necessária e **insuficiente**: um texto que repete a mesma palavra em toda frase é
altamente coeso e pode não dizer nada.

## O mapa das técnicas

### 1. Cadeias lexicais e sobreposição

**Fundamentação.** Texto coerente reencontra os mesmos referentes. A versão mais simples conta
palavras compartilhadas entre frases vizinhas; a versão séria constrói **cadeias lexicais** usando
uma ontologia (WordNet) para ligar sinônimos e hiperônimos.

**Limitações:** sem ontologia, sinonímia é invisível — e boa escrita **evita** repetir.

**Referência:** Morris & Hirst (1991).

### 2. Entity grid

**Fundamentação.** A técnica clássica. Monta-se uma matriz **entidades × frases**, onde cada
célula guarda o papel sintático da entidade naquela frase: **S** (sujeito), **O** (objeto),
**X** (outro), **-** (ausente). Texto coerente produz **transições suaves** (S→S, S→O); texto
incoerente produz entidades que aparecem e somem. A distribuição das transições vira um vetor de
características.

**Quando usar:** ordenação de sentenças, avaliação de sumários. **Limitação decisiva:** o método
original depende de **resolução de correferência** para saber que "a equipe" e "o grupo" são a
mesma entidade. Sem isso, ele só enxerga repetição literal — e vira a técnica 1 com passos extras.

**Referência:** Barzilay & Lapata (2005, 2008); Guinaudeau & Strube (2013) para a variante em grafo.

### 3. Similaridade semântica entre frases vizinhas

**Fundamentação.** Se frases consecutivas falam de coisas próximas, seus vetores são próximos.
Resolve a sinonímia que derruba a técnica 1. **Limitações:** mede *proximidade temática*, não
*progressão* — e por isso premia texto que fica repetindo o mesmo assunto sem avançar.

### 4. Modelo de linguagem condicional

**Fundamentação.** Quanto o modelo esperava a frase *i* dadas as anteriores? Continuação coerente
é mais previsível que salto de assunto. Parente da tarefa de *next sentence prediction*.

**Limitações:** mede **previsibilidade**, e o texto mais previsível que existe é o repetitivo.

**Referência:** Devlin et al. (2019) para NSP; Li & Jurafsky (2017) para modelos neurais de coerência.

## Como se valida um medidor de coerência

Não há rótulo humano barato, então a área usa **perturbações controladas** — a mesma lógica do
par mínimo, aplicada a texto:

| protocolo | construção | critério |
|---|---|---|
| **discriminação (shuffle)** | original vs. permutações | o original deve pontuar acima |
| **inserção** | mover uma frase de lugar | o original deve vencer as *n* variantes |
| **ordenação** | reconstruir a ordem original | correlação de Kendall com a verdadeira |

O teste de embaralhamento é o mais usado porque as permutações têm **exatamente as mesmas
palavras** — o que muda é só a ordem. Qualquer diferença de pontuação é atribuível ao
encadeamento.

**Atenção a um detalhe que invalida resultados:** se a métrica der o **mesmo valor** para muitas
permutações, o "1º lugar" é um empate, não uma vitória. A célula abaixo conta os empates no topo
justamente por isso.

In [12]:
PARAGRAFOS = {
 "coerente_A": [
   "A equipe instalou sensores de temperatura na floresta em janeiro.",
   "Os aparelhos registraram medições a cada quinze minutos.",
   "Esses registros revelaram uma variação diária maior que a esperada.",
   "Por isso, o grupo decidiu revisar o modelo climático que usava.",
 ],
 "coerente_B": [
   "A biblioteca recebeu uma doação de mil volumes no mês passado.",
   "O acervo precisou ser catalogado antes de entrar em circulação.",
   "Dois funcionários trabalharam nisso durante três semanas.",
   "As obras já estão disponíveis para empréstimo desde segunda-feira.",
 ],
 "sem_relacao": [
   "A equipe instalou sensores de temperatura na floresta em janeiro.",
   "O preço do café subiu bastante no mercado internacional.",
   "Muitos gatos preferem dormir durante a tarde.",
   "A biblioteca municipal fecha às dezoito horas.",
 ],
 "coeso_vazio": [
   "Os sensores são equipamentos importantes.",
   "Os sensores foram instalados pela equipe.",
   "Os sensores registram dados.",
   "Os sensores são muito utilizados.",
 ],
}
PARAGRAFOS["embaralhado_A"] = [PARAGRAFOS["coerente_A"][i] for i in (2, 0, 3, 1)]
PARAGRAFOS["embaralhado_B"] = [PARAGRAFOS["coerente_B"][i] for i in (3, 1, 0, 2)]


substantivos = lambda fr: {t.lemma_.lower() for t in nlp(fr) if t.pos_ in ("NOUN", "PROPN")}

# ---- C1: sobreposição lexical entre frases vizinhas ----
def C1_lexical(fs):
    ov = [len(substantivos(fs[i]) & substantivos(fs[i + 1])) for i in range(len(fs) - 1)]
    return sum(1 for o in ov if o) / len(ov)


# ---- C2: entity grid (Barzilay & Lapata), sem resolução de correferência ----
PRIORIDADE = {"S": 3, "O": 2, "X": 1, "-": 0}

def grid_entidades(fs):
    grid = defaultdict(lambda: ["-"] * len(fs))
    for i, fr in enumerate(fs):
        for t in nlp(fr):
            if t.pos_ not in ("NOUN", "PROPN"):
                continue
            papel = ("S" if t.dep_ in ("nsubj", "nsubj:pass")
                     else "O" if t.dep_ in ("obj", "iobj", "obl") else "X")
            e = t.lemma_.lower()
            if PRIORIDADE[papel] > PRIORIDADE[grid[e][i]]:
                grid[e][i] = papel
    return dict(grid)

def C2_entity_grid(fs):
    # fração de transições em que a entidade PERMANECE presente entre frases vizinhas
    trans = Counter()
    for linha in grid_entidades(fs).values():
        for i in range(len(linha) - 1):
            trans[(linha[i], linha[i + 1])] += 1
    total = sum(trans.values()) or 1
    return sum(v for (a, b), v in trans.items() if a != "-" and b != "-") / total


# ---- C3: similaridade de embeddings entre frases vizinhas ----
def C3_embeddings(fs):
    return sum(float(F.cosine_similarity(embedding(fs[i]), embedding(fs[i + 1]), dim=0))
               for i in range(len(fs) - 1)) / (len(fs) - 1)


# ---- C4: log-prob condicional ----
def _lp_cond(contexto, frase):
    n = tok_q(contexto, return_tensors="pt")["input_ids"].shape[1] if contexto else 0
    enc = tok_q(contexto + frase, return_tensors="pt"); ids = enc["input_ids"][0]
    with torch.no_grad():
        lp = torch.log_softmax(mod_q(**enc).logits, dim=-1)
    v = [float(lp[0, i - 1, ids[i]]) for i in range(max(n, 1), len(ids))]
    return sum(v) / len(v)

def C4_lm_condicional(fs):
    return sum(_lp_cond(" ".join(fs[:i]) + " ", fs[i]) for i in range(1, len(fs))) / (len(fs) - 1)


print("grid de entidades do parágrafo COERENTE (S=sujeito, O=objeto, X=outro, -=ausente):")
for ent, linha in grid_entidades(PARAGRAFOS["coerente_A"]).items():
    print(f"  {ent:14} {linha}")

grid de entidades do parágrafo COERENTE (S=sujeito, O=objeto, X=outro, -=ausente):
  equipe         ['S', '-', '-', '-']
  sensor         ['O', '-', '-', '-']
  temperatura    ['X', '-', '-', '-']
  floresta       ['O', '-', '-', '-']
  janeiro        ['O', '-', '-', '-']
  aparelho       ['-', 'S', '-', '-']
  medição        ['-', 'O', '-', '-']
  minuto         ['-', 'X', '-', '-']
  registro       ['-', '-', 'S', '-']
  variação       ['-', '-', 'O', '-']
  grupo          ['-', '-', '-', 'S']
  modelo         ['-', '-', '-', 'O']


In [13]:
# ---- COMPARAÇÃO: Coerência ----
print("--- custo por parágrafo ---")
for nome, fn in [("C1 lexical", C1_lexical), ("C2 entity grid", C2_entity_grid),
                 ("C3 embeddings", C3_embeddings), ("C4 LM condicional", C4_lm_condicional)]:
    print(f"  {nome:20} {custo_ms(fn, PARAGRAFOS['coerente_A'], 2):9.1f} ms")

print(f"\n{'parágrafo':16} {'C1 lex':>8} {'C2 grid':>9} {'C3 emb':>9} {'C4 LM':>9}")
for n, fs in PARAGRAFOS.items():
    print(f"{n:16} {C1_lexical(fs):8.2f} {C2_entity_grid(fs):9.2f} "
          f"{C3_embeddings(fs):9.3f} {C4_lm_condicional(fs):9.3f}")

print("\n--- TESTE DE EMBARALHAMENTO: o original vence as 24 permutações? ---")
print("(empates no topo revelam métrica que não discrimina nada)\n")
print(f"{'parágrafo':13} {'técnica':18} {'posição':>9} {'empatados no topo':>19}")
for chave in ["coerente_A", "coerente_B"]:
    original = PARAGRAFOS[chave]
    for nome, fn in [("C1 lexical", C1_lexical), ("C2 entity grid", C2_entity_grid),
                     ("C3 embeddings", C3_embeddings), ("C4 LM condicional", C4_lm_condicional)]:
        ranking = sorted(((fn(list(p)), p) for p in itertools.permutations(original)),
                         key=lambda x: -x[0])
        pos = next(i for i, (s, p) in enumerate(ranking) if list(p) == original) + 1
        empates = sum(1 for s, _ in ranking if abs(s - ranking[0][0]) < 1e-9)
        alerta = "  <- empate total" if empates == len(ranking) else ""
        print(f"{chave:13} {nome:18} {pos:6}º/24 {empates:15}{alerta}")

--- custo por parágrafo ---
  C1 lexical                35.6 ms
  C2 entity grid            24.1 ms
  C3 embeddings            312.6 ms
  C4 LM condicional       4082.0 ms

parágrafo          C1 lex   C2 grid    C3 emb     C4 LM
coerente_A           0.00      0.00     0.671    -1.970
coerente_B           0.00      0.00     0.671    -2.132
sem_relacao          0.00      0.00     0.548    -2.975
coeso_vazio          1.00      0.33     0.780    -1.932
embaralhado_A        0.00      0.00     0.640    -2.326
embaralhado_B        0.00      0.00     0.696    -2.263

--- TESTE DE EMBARALHAMENTO: o original vence as 24 permutações? ---
(empates no topo revelam métrica que não discrimina nada)

parágrafo     técnica              posição   empatados no topo
coerente_A    C1 lexical              1º/24              24  <- empate total
coerente_A    C2 entity grid          1º/24              24  <- empate total
coerente_A    C3 embeddings           5º/24               2
coerente_A    C4 LM condicion

### Lendo a comparação — Coerência

Esta é a dimensão em que o estudo mais claramente **não encontra uma técnica que funcione**, e o
mais útil aqui é entender por quê.

**C1 e C2 são inertes: dão 0,00 aos dois parágrafos coerentes e 0,00 aos embaralhados.** Zero
discriminação. O grid de entidades impresso na célula anterior mostra a causa de forma
inescapável: **cada entidade aparece em exatamente uma frase.** `equipe`, `sensor`, `aparelho`,
`medição`, `registro`, `grupo`, `modelo` — nenhuma linha tem dois valores preenchidos.

E isso acontece porque o parágrafo é **bem escrito**: `equipe`→`grupo`, `sensores`→`aparelhos`→
`medições`→`registros` são cadeias de referência **semânticas**, e o casamento por lema não
enxerga nenhuma delas. O único parágrafo que pontua é `coeso_vazio` (C1 = 1,00, C2 = 0,33) —
aquele que repete "os sensores" quatro vezes e não diz nada.

**As duas técnicas premiam exatamente o vício que um revisor marcaria primeiro.** E o entity grid
não está mal implementado: está implementado **sem resolução de correferência**, que é justamente
o componente que Barzilay & Lapata usavam. Sem ele, o grid é a sobreposição lexical com mais
passos — o que a igualdade dos resultados confirma.

**O teste de embaralhamento expõe o problema que uma tabela de médias esconderia.** C1 e C2
aparecem em "1º/24" — o que, lido sem cuidado, pareceria vitória. A coluna de empates mostra
**24 empatados no topo**: todas as permutações recebem a mesma nota, o "primeiro lugar" é o acaso
da ordenação. Sem essa coluna, o estudo teria reportado duas técnicas perfeitas que na verdade não
medem nada.

**C4 é a única que às vezes funciona — e o segundo parágrafo desmente o primeiro.**

| parágrafo | C1 | C2 | C3 | C4 |
|---|---|---|---|---|
| `coerente_A` | 1º (24 empates) | 1º (24 empates) | 5º/24 | **1º/24** |
| `coerente_B` | 1º (24 empates) | 1º (24 empates) | 13º/24 | **9º/24** |

Em `coerente_A`, o log-prob condicional coloca a ordem original em 1º entre 24 — resultado forte.
Em `coerente_B`, a mesma técnica coloca em **9º**. Não há nada de errado com o parágrafo B; ele
simplesmente tem um encadeamento que o modelo não prevê tão bem.

**Esta é a lição metodológica mais importante do notebook.** Se o estudo tivesse testado só o
parágrafo A — como é comum, e como a versão anterior deste material fazia — teria concluído que a
técnica funciona. Um segundo item derrubou a conclusão. **n = 1 não é evidência**, e a diferença
entre "demonstração" e "avaliação" é exatamente essa.

**C3 fica em 5º e 13º**: em torno do acaso, com o custo de um modelo carregado. E na tabela de
pontuações ele comete o mesmo erro dos outros — dá **0,780** ao `coeso_vazio`, a maior nota de
todas, porque frases que repetem o mesmo assunto têm vetores próximos.

**Custo:** C2 é a mais barata (~15 ms) e C4 a mais cara (~2 200 ms) — **cerca de 150×** mais. A
técnica mais cara é a única com algum sinal, e mesmo ela acerta um parágrafo em dois.

**Conclusão da dimensão:** com os recursos aqui — sem correferência, sem parser de discurso, sem
modelo de coerência treinado — **não há técnica confiável**. O caminho sério passa por resolução
de correferência (para o grid funcionar), análise de discurso (RST/PDTB) e modelos treinados na
tarefa. Reportar qualquer um destes quatro números como "medida de coerência" seria
irresponsável.

---
# Validação: como se sabe que um avaliador funciona

As tabelas acima só significam alguma coisa porque o corpus tem **gabarito**. Sem saber qual era o
defeito de cada frase, "6/6" e "0/6" seriam números sem referente. Os instrumentos da área:

| instrumento | o que responde | quando usar |
|---|---|---|
| **acurácia em pares mínimos** | a técnica ordena certo duas frases que diferem em uma coisa? | validar pontuadores contínuos |
| **taxa de detecção + falsos positivos** | dispara no alvo? dispara à toa? | validar detectores binários |
| **MCC (Matthews)** | desempenho com classes desbalanceadas | classificação de aceitabilidade |
| **Spearman / Pearson / Kendall** | a ordem produzida bate com a humana? | correlação com juízo humano |
| **Kappa de Cohen / alpha de Krippendorff** | os anotadores humanos concordam entre si? | **o teto** de qualquer métrica |

A última linha é a mais esquecida: **se humanos não concordam entre si sobre um fenômeno, nenhuma
métrica automática pode correlacionar bem com "a" resposta humana** — ela não existe. O acordo
entre anotadores é o limite superior do que se pode exigir.

**Benchmarks e recursos** — para inglês: **CoLA** (aceitabilidade), **BLiMP** (pares mínimos por
fenômeno linguístico), **GUM** (discurso anotado). Para **português**: **ASSIN** e **ASSIN 2**
(similaridade semântica e inferência), os corpora e ferramentas do **NILC/USP** (incluindo o
**Coh-Metrix-Port**), e as tarefas compartilhadas do **PROPOR**.

**Duas armadilhas que este estudo encontrou na prática:**

1. **Empates disfarçados de vitória.** Uma métrica constante fica em "1º lugar" num ranking, e
   isso não significa nada. Conte os empates.
2. **`n = 1` que vira conclusão.** Uma técnica que funciona no primeiro exemplo pode falhar no
   segundo — aconteceu com C4. Todo resultado aqui vem de 6 bases ou 2 parágrafos, e mesmo isso é
   pouco.

In [14]:
# ---- TABELA CONSOLIDADA ----
linhas = [
 ("Gramática", "T1 parser (spaCy)",       "detector",  "~4 ms",     "conc_verbal 6/6, conc_nominal 5/6", "sim, token+traço", "ortografia 0/6"),
 ("Gramática", "T2 LanguageTool",         "detector",  "~0.4 s",   "ortografia 6/6, conc_verbal 1/6",   "sim, com regra",   "cobertura pt fraca"),
 ("Gramática", "T3 SLOR (causal)",        "pontuador", "~0.5 s",   "29/30 pares",                       "não",              "infla com palavra OOV"),
 ("Gramática", "T4 PLL (mascarado)",      "pontuador", "~0.4 s",   "30/30 pares",                       "não",              "pune vocabulário raro"),
 ("Estrutura", "E1 núcleo predicativo",   "detector",  "~3.5 ms",     "fragmento 6/6, 0 falso positivo",   "sim, raiz",        "só vê completude"),
 ("Estrutura", "E2 complexidade",         "pontuador", "~4 ms",     "prof/orações separam fragmento",    "não",              "ramificação é constante"),
 ("Estrutura", "E3 dep. length",          "pontuador", "~4 ms",     "fragmento 6/6, resto ~0",           "não",              "redundante com E1"),
 ("Estrutura", "E4 legibilidade",         "pontuador", "~0.01 ms",  "0-1/6 (acaso)",                     "não",              "anticorrelacionada"),
 ("Semântica", "S1 surpresa",             "pontuador", "~0.5 s",   "anomalia 4/6",                      "sim, token",       "raro = anômalo"),
 ("Semântica", "S2 verbo mascarado",      "pontuador", "~38 ms",   "anomalia 5/6",                      "sim, + esperados", "n/a sem verbo raiz"),
 ("Semântica", "S3 embeddings",           "pontuador", "~33 ms",    "ordem não confiável",               "não",              "cego à negação"),
 ("Semântica", "S4 LLM",                  "detector",  "~1.6 s",  "anomalia 4/6",                      "sim, texto",       "aceita metáfora"),
 ("Coerência", "C1 lexical",              "pontuador", "~25 ms",    "empate total (24/24)",              "não",              "cego a sinônimo"),
 ("Coerência", "C2 entity grid",          "pontuador", "~15 ms",    "empate total (24/24)",              "não",              "exige correferência"),
 ("Coerência", "C3 embeddings",           "pontuador", "~0.2 s",   "5º e 13º de 24",                    "não",              "premia repetição"),
 ("Coerência", "C4 LM condicional",       "pontuador", "~2.2 s",  "1º e 9º de 24",                     "não",              "premia previsível"),
]
cab = ("dimensão", "técnica", "tipo", "custo", "desempenho", "localiza?", "limitação principal")
larg = [11, 24, 10, 10, 35, 19, 24]
print("".join(f"{c:<{w}}" for c, w in zip(cab, larg)))
print("-" * sum(larg))
print("(custos medidos nesta execução, CPU; variam entre rodadas - leia a ordem de grandeza)\n")
ult = None
for l in linhas:
    if ult and l[0] != ult: print()
    ult = l[0]
    print("".join(f"{str(v):<{w}}" for v, w in zip(l, larg)))

dimensão   técnica                 tipo      custo     desempenho                         localiza?          limitação principal     
-------------------------------------------------------------------------------------------------------------------------------------
(custos medidos nesta execução, CPU; variam entre rodadas - leia a ordem de grandeza)

Gramática  T1 parser (spaCy)       detector  ~4 ms     conc_verbal 6/6, conc_nominal 5/6  sim, token+traço   ortografia 0/6          
Gramática  T2 LanguageTool         detector  ~0.4 s    ortografia 6/6, conc_verbal 1/6    sim, com regra     cobertura pt fraca      
Gramática  T3 SLOR (causal)        pontuador ~0.5 s    29/30 pares                        não                infla com palavra OOV   
Gramática  T4 PLL (mascarado)      pontuador ~0.4 s    30/30 pares                        não                pune vocabulário raro   

Estrutura  E1 núcleo predicativo   detector  ~3.5 ms   fragmento 6/6, 0 falso positivo    sim, raiz         

# Síntese

## A fronteira custo × qualidade

O resultado mais consistente do estudo é que **custo não compra qualidade**. Em três das quatro
dimensões, a técnica mais barata é a melhor ou empata com a melhor:

- **Gramática:** o parser (~4 ms) vence a ferramenta madura (~0,4 s) em concordância verbal por
  **6/6 contra 1/6** — e perde de 0/6 para 6/6 em ortografia. Complementares, não ordenáveis.
- **Estrutura:** o teste da raiz (~3,5 ms) é o melhor detector do estudo. A legibilidade, centenas
  de vezes mais barata, é a pior técnica de todas — **anticorrelacionada** com a correção.
- **Semântica:** o verbo mascarado (~38 ms) supera o LLM (~1,6 s) por 5/6 contra 4/6, custando
  **40× menos**, e ainda entrega a lista do que o modelo esperava naquela posição.
- **Coerência:** a mais cara (~2,2 s) é a única com algum sinal — e acerta um parágrafo em dois.

## Guia de escolha

1. **Erro de forma?** Parser para concordância, detecção lexical para grafia, teste da raiz para
   fragmento. Determinísticos, ~4 ms, localizados. Não use LLM aqui: custa 500× mais e acerta
   menos.
2. **Precisa ordenar frases por aceitabilidade?** PLL dentro de pares mínimos; SLOR entre frases
   de conteúdos diferentes. Nenhum dos dois em escala absoluta.
3. **Anomalia semântica?** Mascare a posição que o parser indicar. É a única combinação
   linguística+estatística do estudo, e é por isso que ela ganha.
4. **Coerência?** Nenhuma técnica aqui é confiável. Antes de medir, obtenha **resolução de
   correferência**; sem ela o entity grid é decorativo.
5. **Vai comparar técnicas?** Construa pares mínimos com gabarito, conte empates, e use mais de um
   item por fenômeno.

## O que este estudo não cobre

Honestidade sobre o alcance: **6 bases e 2 parágrafos** demonstram mecanismos, não estimam
desempenho — intervalos de confiança sobre 6 observações não significam nada. Faltaram as técnicas
que exigem treinamento (classificador de aceitabilidade estilo CoLA, modelo neural de coerência,
NLI ajustado para português) e as que exigem recursos ausentes (correferência, parser de
constituintes para Yngve/Frazier, parser de discurso). O juiz LLM é de 1,5B, muito abaixo do que
se usa a sério. E não há **nenhuma anotação humana**: o gabarito é meu, sobre defeitos que eu
mesmo injetei, o que mede detecção de defeitos sintéticos — não qualidade de texto real.

## Três lições que atravessam as quatro dimensões

- **A comparação controlada é o único terreno firme.** Todo número confiável aqui veio de comparar
  itens que diferem numa coisa só. Valores absolutos — SLOR, PLL, Flesch, cosseno — não têm escala
  interpretável.
- **Métricas que não podem variar, e empates que parecem vitória.** `ramificacao` = 1,00 sempre;
  `cruzamentos` = 0 sempre; C1 e C2 empatando as 24 permutações. Três casos em que a coluna existe
  e a informação não. Sempre verifique se a métrica **varia** antes de interpretá-la.
- **Combinar camadas ganha de escalar uma só.** A melhor técnica semântica (S2) é a que usa o
  parser para decidir onde perguntar ao modelo. A pior é a que só pergunta mais alto (S4, um LLM
  inteiro). Estrutura linguística e estatística não competem — se complementam.

## Referências

**Gramática e aceitabilidade** — Naber (2003), *A Rule-Based Style and Grammar Checker*;
Nivre et al. (2020), *Universal Dependencies v2*; Pauls & Klein (2012); Lau, Clark & Lappin (2017),
*Grammaticality, Acceptability, and Probability*; Wang & Cho (2019); Salazar et al. (2020),
*Masked Language Model Scoring*; Warstadt, Singh & Bowman (2019), *CoLA*; Warstadt et al. (2020),
*BLiMP*.

**Estrutura e legibilidade** — Yngve (1960); Frazier (1985); Gibson (1998); Flesch (1948);
Gunning (1952); Martins et al. (1996); Graesser et al. (2004), *Coh-Metrix*; Scarton & Aluísio
(2010), *Coh-Metrix-Port*; Lu (2010); Futrell, Mahowald & Gibson (2015).

**Semântica** — Harris (1954); Erk (2007); Bowman et al. (2015), *SNLI*; Reimers & Gurevych (2019),
*Sentence-BERT*; Real et al. (2020), *ASSIN 2*; Sasano & Korhonen (2020).

**Coerência** — Morris & Hirst (1991); Barzilay & Lapata (2005, 2008), *Modeling Local Coherence*;
Guinaudeau & Strube (2013); Li & Jurafsky (2017); Devlin et al. (2019).

**Modelos usados** — Souza, Nogueira & Lotufo (2020), *BERTimbau*; Qwen2.5 (2024);
Honnibal & Montani, *spaCy*.